# TP1 - Compteur de mots dans les fichiers txt

L'objectif de ce travail est de mettre en œuvre trois approches visant à compter le nombre d'occurrences de chaque mot dans un fichier .txt, tout en comprenant le fonctionnement de chacune d'elles.

Les 3 versions proposées sont :

- Version simple utilisant Python

- Pseudo-MapReduce

- Hadoop MapReduce

Pour évaluer chaque approche, le temps d'exécution de chacune sera calculé en utilisant des fichiers texte contenant des nombres de mots différents.

## Importation des bibliothèques utilisées

In [17]:
from pathlib import Path
import random
from time import perf_counter
from collections import defaultdict

## Création des fichiers txt

Quatre fichiers .txt contenant des mots aléatoires sur chaque ligne ont été créés. Chaque fichier contient dix fois plus de mots que le précédent, dans le but d'évaluer les performances de chaque approche pour différentes tailles de fichiers d'entrée.

In [ ]:
n = 1000
tailles = [1, 10, 100, 1000]
mots = ["chat", "chien", "arbre", "fleur", "maison", "soleil", "montagne", "livre", "ciel", "mer", "lune", "route", "oiseau", "jardin", "voiture"]

dossier = Path("data")
dossier.mkdir(exist_ok=True)

fichiers = {}

for m in tailles:
    lignes = n * m
    chemin = dossier / f"mots_taille_{m}.txt"

    with chemin.open("w", encoding="utf-8") as fichier:
        for i in range(lignes):
            mot = random.choice(mots)
            fichier.write(mot + "\n")

    fichiers[m] = chemin

    print(f"{chemin.name} : {lignes} lignes, {chemin.stat().st_size} bytes")

mots_taille_1.txt : 1000 lignes, 6309 bytes
mots_taille_10.txt : 10000 lignes, 62711 bytes
mots_taille_100.txt : 100000 lignes, 626776 bytes
mots_taille_1000.txt : 1000000 lignes, 6265995 bytes


## Version simple utilisant Python

In [15]:
def compter_occurrences(fichier):
    with open(fichier, 'r', encoding = 'utf-8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur [mot] = compteur.get(mot, 0) + 1
    return compteur

if __name__ == "__main__":
    dossier = Path("data")
    fichiers = sorted(dossier.glob("*.txt"))
    for fichier in fichiers:
        start = perf_counter()
        resultats = compter_occurrences(fichier)
        end = perf_counter()
        temps = end - start
        print(f"fichier : {fichier.name}\ntemps : {temps:.6f} secondes\nnombre de mots : {sum(resultats.values())}\nnombre de mots uniques : {len(resultats)},\n\nOccurrences :")

        for mot, nb in sorted(resultats.items(), key = lambda x : x [1], reverse = True):
            print (f"{mot} : {nb}")
        print("\n\n\n")

fichier : mots_taille_1.txt
temps : 0.000606 secondes
nombre de mots : 1000
nombre de mots uniques : 15,

Occurrences :
voiture : 82
lune : 78
livre : 75
montagne : 73
arbre : 73
oiseau : 70
maison : 68
chat : 68
jardin : 66
route : 65
fleur : 61
mer : 59
ciel : 59
chien : 58
soleil : 45




fichier : mots_taille_10.txt
temps : 0.001742 secondes
nombre de mots : 10000
nombre de mots uniques : 15,

Occurrences :
maison : 698
lune : 694
soleil : 688
route : 683
arbre : 676
chien : 667
montagne : 666
livre : 664
jardin : 663
voiture : 661
mer : 655
chat : 653
fleur : 651
ciel : 641
oiseau : 640




fichier : mots_taille_100.txt
temps : 0.014767 secondes
nombre de mots : 100000
nombre de mots uniques : 15,

Occurrences :
chien : 6781
oiseau : 6757
livre : 6746
montagne : 6725
fleur : 6709
arbre : 6689
lune : 6675
chat : 6672
ciel : 6668
maison : 6639
soleil : 6630
route : 6616
jardin : 6592
mer : 6551
voiture : 6550




fichier : mots_taille_1000.txt
temps : 0.116312 secondes
nombre de mot

## Pseudo-MapReduce

In [18]:
def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced [mot] = sum(valeurs)
    return reduced

if __name__ == "__main__":
    dossier = Path("data")
    fichiers = sorted(dossier.glob("*.txt"))
    mapped = []
    for fichier in fichiers:
        start = perf_counter()
        with open(fichier, "r", encoding = "utf-8") as f:
            for ligne in f:
                mapped.extend(mapper(ligne))

        grouped = shuffle(mapped)
        reduced = reducer(grouped)

        end = perf_counter()
        temps = end - start
        print(f"fichier : {fichier.name}\ntemps : {temps:.6f} secondes\nnombre de mots : {sum(reduced.values())}\nnombre de mots uniques : {len(reduced)},\n\nOccurrences :")

        for mot, count in sorted(reduced.items(), key = lambda x : x[1] , reverse = True):
            print (f"{mot} : {count}")
        print("\n\n\n")

fichier : mots_taille_1.txt
temps : 0.000662 secondes
nombre de mots : 1000
nombre de mots uniques : 15,

Occurrences :
voiture : 82
lune : 78
livre : 75
montagne : 73
arbre : 73
oiseau : 70
maison : 68
chat : 68
jardin : 66
route : 65
fleur : 61
mer : 59
ciel : 59
chien : 58
soleil : 45




fichier : mots_taille_10.txt
temps : 0.004814 secondes
nombre de mots : 11000
nombre de mots uniques : 15,

Occurrences :
lune : 772
maison : 766
arbre : 749
route : 748
voiture : 743
livre : 739
montagne : 739
soleil : 733
jardin : 729
chien : 725
chat : 721
mer : 714
fleur : 712
oiseau : 710
ciel : 700




fichier : mots_taille_100.txt
temps : 0.048347 secondes
nombre de mots : 111000
nombre de mots uniques : 15,

Occurrences :
chien : 7506
livre : 7485
oiseau : 7467
montagne : 7464
lune : 7447
arbre : 7438
fleur : 7421
maison : 7405
chat : 7393
ciel : 7368
route : 7364
soleil : 7363
jardin : 7321
voiture : 7293
mer : 7265




fichier : mots_taille_1000.txt
temps : 0.501016 secondes
nombre de mot

## Hadoop MapReduce